In [ ]:
# Ô 0-1: gói (trl 0.29.1 - transformers 5.18.0 - peft 0.21.1, gỡ torchao — cùng bộ đã chạy lượt 299/304) + Java cho SPICE
import subprocess, sys, os, time, glob, json, shutil, re
T0 = time.time()
def sh(c):
    r = subprocess.run(c, shell=True, capture_output=True, text=True); print((r.stdout + r.stderr)[-1500:], flush=True); return r.returncode
sh("apt-get -qq update && apt-get -qq install -y openjdk-8-jre-headless")
sh("update-alternatives --set java /usr/lib/jvm/java-8-openjdk-amd64/jre/bin/java; java -version")
sh(f"{sys.executable} -m pip install -q 'transformers==5.18.0' 'peft==0.21.1' 'trl==0.29.1' accelerate bitsandbytes pycocoevalcap sacrebleu")
sh(f"{sys.executable} -m pip uninstall -y torchao")
sh(f'{sys.executable} -c "import torch, transformers, peft, trl; print(transformers.__version__, peft.__version__, trl.__version__, torch.cuda.device_count())"')
import torch
assert torch.cuda.device_count() == 2, "⛔ cần T4 x2 - đổi Accelerator rồi chạy lại"

In [ ]:
# Ô 0-2: đường dẫn, chép mã, hoà S1, hàm chạy nền
W = "/kaggle/working"; RA = f"{W}/ra"; M = f"{W}/s1_merged"
def tim(duoi):
    for sau in range(1, 6):
        g = glob.glob("/kaggle/input/" + "*/" * sau + duoi)
        if g: return g[0]
    raise FileNotFoundError(duoi)

SRC = os.path.dirname(tim("ra_gen.py")); D312 = os.path.dirname(tim("ra_grpo_g0.py"))
BUNDLE = next(r for r, d, f in os.walk("/kaggle/input") if "adapter_s1_seed101" in d and "images" in d)
shutil.copytree(SRC, RA, dirs_exist_ok=True)
for f in os.listdir(D312): shutil.copy(f"{D312}/{f}", RA)
print("SRC", SRC, "\nD312", D312, "\nBUNDLE", BUNDLE)
r = subprocess.run(["python", "grpo_spice.py", "--merge", "--bundle", BUNDLE, "--merged", M], cwd=RA, capture_output=True, text=True)
print((r.stdout + r.stderr)[-1500:]); assert r.returncode == 0
V = dict(recs=f"{BUNDLE}/p1_val_rows.jsonl", ocr=f"{BUNDLE}/ocr.jsonl", img=f"{BUNDLE}/images", ex=f"{SRC}/ex_val_k4.jsonl")
ENV = {**os.environ, "TQDM_DISABLE": "1", "HF_HUB_DISABLE_PROGRESS_BARS": "1", "PYTHONUNBUFFERED": "1",
       "PYTORCH_ALLOC_CONF": "expandable_segments:True", "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}
def chay(script, args, log, gpu):
    return subprocess.Popen(["python", f"{RA}/{script}", *args], stdout=open(log, "w"), stderr=subprocess.STDOUT,
                            env={**ENV, "CUDA_VISIBLE_DEVICES": str(gpu)}, cwd=RA, start_new_session=True)
def doi(ps, logs):
    while any(p.poll() is None for p in ps):
        time.sleep(120); print(time.strftime("%H:%M"), f"{(time.time()-T0)/3600:.2f} h", [open(l).read().splitlines()[-1:] for l in logs], flush=True)
    assert all(p.returncode == 0 for p in ps), [open(l).read()[-2000:] for l in logs]
def py(*a):
    r = subprocess.run(["python", *a], capture_output=True, text=True, cwd=RA); print(r.stdout[-5000:], r.stderr[-2000:]); assert r.returncode == 0

In [ ]:
# Ô T-2: chép điểm lưu, đệm khối, cấu hình từ output lần trước (lần 1 không có gì)
OUT = f"{W}/ra_grpo"; os.makedirs(OUT, exist_ok=True)
for s in range(1, 6):
    for p in glob.glob("/kaggle/input/" + "*/" * s + "ra_grpo/checkpoint-*"):
        if os.path.isdir(p): shutil.copytree(p, f"{OUT}/{os.path.basename(p)}", dirs_exist_ok=True)
    for f in ("ra_cache.json", "cau_hinh.json"):
        for c in glob.glob("/kaggle/input/" + "*/" * s + f"ra_grpo/{f}"): shutil.copy(c, OUT)
print("trong OUT:", sorted(os.listdir(OUT))[-6:])

# Ô T-3a: (chỉ khi chưa có cau_hinh.json) thăm dò 20 bước trên 50 câu nhắc DÀI NHẤT — đo VRAM, s/bước; tràn thì lùi micro-batch
CH = f"{OUT}/cau_hinh.json"
# 312-WSL: thử cấu hình ck500 đã chạy hết 500 bước TRƯỚC (bs 4 từng OOM ở bước ~443 dù probe 13,22 GiB)
CFG = (["--bs", "2", "--accum", "8", "--gen-chunk", "8"], ["--bs", "4", "--accum", "4"])
if not os.path.exists(CH):
    for cfg in CFG:
        shutil.rmtree(f"{W}/probe", ignore_errors=True)
        cmd = ["python", f"{RA}/ra_grpo.py", "--kho", f"{SRC}/train_tru_val.jsonl", "--gio", "3", "--",
               "--probe", "--no-q4", *cfg, "--bundle", BUNDLE, "--merged", M, "--out", f"{W}/probe"]
        r = subprocess.run(cmd, cwd=RA, stdout=open(f"{W}/probe_{cfg[1]}.log", "w"), stderr=subprocess.STDOUT,
                           env={**ENV, "CUDA_VISIBLE_DEVICES": "0"})
        L = open(f"{W}/probe_{cfg[1]}.log", errors="ignore").read()
        xong = [l for l in L.splitlines() if l.startswith("[xong]")]
        print(cfg, "mã thoát", r.returncode, xong[-1:] or L[-1500:], flush=True)
        print("\n".join(l for l in L.splitlines() if l.startswith(("[kho]", "[RA]", "[câu nhắc]", "[LoRA mới]", "[chép]")))[:3000])
        if r.returncode == 0 and xong:
            vram = float(re.search(r"đỉnh VRAM ([\d.]+)", xong[-1]).group(1)); spb = float(re.search(r"([\d.]+) s/bước", xong[-1]).group(1))
            if vram <= 14.3:
                json.dump(dict(cfg=cfg, vram=vram, s_buoc=spb), open(CH, "w")); break
    assert os.path.exists(CH), "⛔ cả hai cấu hình đều tràn hoặc lỗi — gửi probe_*.log cho chat"
    if os.path.exists(f"{W}/probe/ra_cache.json"): shutil.copy(f"{W}/probe/ra_cache.json", OUT)
    shutil.rmtree(f"{W}/probe", ignore_errors=True)
print("cấu hình:", json.load(open(CH)))

In [ ]:
# Ô T-3b: train — đúng lệnh train của ck500, bọc bởi ra_grpo.py, tự dừng êm trước 11 h
cfg = json.load(open(CH))["cfg"]
gio = 10.8 - (time.time() - T0) / 3600
cmd = ["python", f"{RA}/ra_grpo.py", "--kho", f"{SRC}/train_tru_val.jsonl", "--gio", f"{gio:.2f}", "--",
       "--train", "--no-q4", *cfg, "--bundle", BUNDLE, "--merged", M, "--out", OUT, "--resume", "auto"]
p = subprocess.Popen(cmd, cwd=RA, stdout=open(f"{W}/train.log", "a"), stderr=subprocess.STDOUT,
                     env={**ENV, "CUDA_VISIBLE_DEVICES": "0"}, start_new_session=True)
while p.poll() is None:
    time.sleep(300)
    L = open(f"{W}/train.log", errors="ignore").read().splitlines()
    sp = [l for l in L if l.startswith("[spice]")][-1:]; cp = [l for l in L if l.startswith("[chép]")][-20:]
    ck = sorted(glob.glob(f"{OUT}/checkpoint-*"), key=lambda s: int(s.rsplit("-", 1)[1]))
    chep = sum(int(re.search(r"(\d+)/", l).group(1)) for l in cp); dung = sum(int(re.search(r"chuẩn (\d+)", l).group(1)) for l in cp)
    print(f"{(time.time()-T0)/3600:.2f} h - lưu cuối {os.path.basename(ck[-1]) if ck else '-'} - "
          f"{sp[0][:110] if sp else (L[-1][:140] if L else '')} - chép 20 lần gần nhất {chep} (đúng {dung})", flush=True)
print("mã thoát", p.returncode)
print("\n".join(open(f"{W}/train.log", errors="ignore").read().splitlines()[-30:]))
for d in (M, RA): shutil.rmtree(d, ignore_errors=True)